# 02 — Data Preparation
**Prague Stay Management**

**Cíl:** Připravit čistá, obohacená a syntetická data pro KPI analýzu. Výstupy tohoto notebooku jsou načteny do DuckDB.

**Kroky:**
1. Čištění a obohacení listings → `listings_enriched.csv`
2. Čištění calendar
3. Generování syntetických dat → `bookings.csv`, `expenses.csv`
4. Načtení všech tabulek do DuckDB

---

## 0. Importy a nastavení

In [25]:
import pandas as pd
import numpy as np
import duckdb
import uuid
import random
import os
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

random.seed(42)
np.random.seed(42)

RAW       = '../data/raw/'
PROCESSED = '../data/processed/'
SYNTHETIC = '../data/synthetic/'
DB_PATH   = '../database/prague_stay.duckdb'

os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(SYNTHETIC, exist_ok=True)
os.makedirs('../database', exist_ok=True)

print('Importy OK')

Importy OK


---
## 1. Čištění a obohacení listings

In [26]:
if not os.path.exists(PROCESSED + 'listings_enriched.csv'):

    # --- 1.1 Načtení a výběr sloupců ---
    listings_raw = pd.read_csv(RAW + 'listings.csv')
    print(f'Načteno: {listings_raw.shape[0]:,} řádků × {listings_raw.shape[1]} sloupců')

    KEEP_COLS = [
        'id', 'name', 'latitude', 'longitude',
        'neighbourhood_cleansed', 'room_type',
        'accommodates', 'bedrooms', 'beds', 'bathrooms_text',
        'price', 'minimum_nights', 'maximum_nights',
        'availability_365', 'estimated_occupancy_l365d', 'estimated_revenue_l365d',
        'number_of_reviews', 'review_scores_rating',
        'host_id', 'host_since', 'host_is_superhost', 'instant_bookable'
    ]
    listings = listings_raw[KEEP_COLS].copy()
    dropped  = set(listings_raw.columns) - set(KEEP_COLS)
    print(f'Po výběru sloupců: {listings.shape[0]:,} řádků × {listings.shape[1]} sloupců')
    print(f'Odstraněno sloupců: {len(dropped)}')
    print(f'  Z toho 100% NaN: license, neighbourhood_group_cleansed, calendar_updated')
    print(f'  Z toho ~59% NaN: neighborhood_overview, neighbourhood')

    # --- 1.2 Konverze a čištění ceny ---
    # POZOR: symbol '$' je artefakt exportu Inside Airbnb.
    # Hodnoty jsou již v CZK — potvrzeno mediánem ~2 152 CZK/noc,
    # což odpovídá reálným cenám na pražském trhu.
    listings['price_czk'] = (
        listings['price']
        .str.replace(r'[\$,]', '', regex=True)
        .astype(float)
    )
    listings.drop(columns=['price'], inplace=True)

    # Dynamický práh outlierů — 99. percentil
    threshold  = listings['price_czk'].quantile(0.99)
    n_outliers = (listings['price_czk'] > threshold).sum()
    listings.loc[listings['price_czk'] > threshold, 'price_czk'] = np.nan
    n_nan_total = listings['price_czk'].isna().sum()
    print(f'\n99. percentil ceny: {threshold:,.0f} CZK')
    print(f'Outlierů nad prahem: {n_outliers} (chybné záznamy nebo blokační ceny)')
    print(f'Celkem NaN v price_czk před imputací: {n_nan_total} ({n_nan_total/len(listings)*100:.1f} %)')

    # --- 1.3 Třístupňová imputace ceny ---
    median_global = listings['price_czk'].median()

    def impute_price(row):
        """Třístupňová imputace: skupina → podskupina → globální medián."""
        if not np.isnan(row['price_czk']):
            return row['price_czk']
        # Úroveň 1: neighbourhood + room_type + accommodates
        m = listings[
            (listings['neighbourhood_cleansed'] == row['neighbourhood_cleansed']) &
            (listings['room_type'] == row['room_type']) &
            (listings['accommodates'] == row['accommodates'])
        ]['price_czk'].median()
        if not np.isnan(m):
            return m
        # Úroveň 2: neighbourhood + room_type
        m = listings[
            (listings['neighbourhood_cleansed'] == row['neighbourhood_cleansed']) &
            (listings['room_type'] == row['room_type'])
        ]['price_czk'].median()
        if not np.isnan(m):
            return m
        # Úroveň 3: globální medián
        return median_global

    median_before        = listings['price_czk'].median()
    listings['price_czk'] = listings.apply(impute_price, axis=1)
    median_after         = listings['price_czk'].median()

    assert listings['price_czk'].isna().sum() == 0, 'Zůstaly NaN v price_czk!'
    low_price = (listings['price_czk'] < 200).sum()
    if low_price > 0:
        print(f'⚠ Ceny < 200 CZK: {low_price} řádků — zkontrolovat')
    print(f'\nMedian před imputací: {median_before:,.0f} CZK')
    print(f'Medián po imputaci:   {median_after:,.0f} CZK')
    print(f'✓ Imputace dokončena, NaN: {listings["price_czk"].isna().sum()}')

    # --- 1.4 Čištění ostatních sloupců ---
    # bedrooms: imputace mediánem podle room_type
    listings['bedrooms'] = listings.groupby('room_type')['bedrooms'].transform(
        lambda x: x.fillna(x.median())
    )
    # beds: imputace mediánem podle accommodates
    listings['beds'] = listings.groupby('accommodates')['beds'].transform(
        lambda x: x.fillna(x.median())
    )
    # review_scores_rating: globální medián
    listings['review_scores_rating'] = listings['review_scores_rating'].fillna(
        listings['review_scores_rating'].median()
    )
    # host_since: konverze na datetime
    listings['host_since'] = pd.to_datetime(listings['host_since'], errors='coerce')
    # bool sloupce
    listings['host_is_superhost'] = listings['host_is_superhost'].map({'t': True, 'f': False})
    listings['instant_bookable']  = listings['instant_bookable'].map({'t': True, 'f': False})

    print('\nZbývající NaN po čištění:')
    remaining = listings.isnull().sum()
    print(remaining[remaining > 0].to_string())

    # --- 1.5 Výpočet vzdálenosti od metra (Haversine — vektorizovaný) ---
    metro = pd.read_csv(RAW + 'metro_stations.csv')

    lats1 = np.radians(listings['latitude'].values)[:, np.newaxis]
    lons1 = np.radians(listings['longitude'].values)[:, np.newaxis]
    lats2 = np.radians(metro['latitude'].values)
    lons2 = np.radians(metro['longitude'].values)

    dlat = lats2 - lats1
    dlon = lons2 - lons1
    a    = np.sin(dlat/2)**2 + np.cos(lats1) * np.cos(lats2) * np.sin(dlon/2)**2
    dist_matrix = 6371000 * 2 * np.arcsin(np.sqrt(a))

    min_indices = np.argmin(dist_matrix, axis=1)
    min_dists   = np.min(dist_matrix, axis=1)

    listings['nearest_metro_id']   = metro['station_id'].iloc[min_indices].values
    listings['nearest_metro_name'] = metro['name'].iloc[min_indices].values
    listings['nearest_metro_line'] = metro['line'].iloc[min_indices].values
    listings['metro_distance_m']   = np.round(min_dists).astype(int)
    listings['near_metro']         = listings['metro_distance_m'] <= 500

    print(f'\nPrůměrná vzdálenost od metra: {listings["metro_distance_m"].mean():.0f} m')
    print(f'Medián vzdálenosti:           {listings["metro_distance_m"].median():.0f} m')
    print(f'near_metro = True  (≤500 m):  {listings["near_metro"].sum():,} ({listings["near_metro"].mean()*100:.1f} %)')
    print(f'near_metro = False (>500 m):  {(~listings["near_metro"]).sum():,} ({(~listings["near_metro"]).mean()*100:.1f} %)')

    # --- Uložení ---
    listings.to_csv(PROCESSED + 'listings_enriched.csv', index=False)
    print(f'\n✓ listings_enriched.csv uložen: {listings.shape[0]:,} řádků × {listings.shape[1]} sloupců')

else:
    listings = pd.read_csv(PROCESSED + 'listings_enriched.csv')
    print(f'listings_enriched.csv již existuje: {len(listings):,} řádků — přeskakuji čištění')

Načteno: 10,834 řádků × 79 sloupců
Po výběru sloupců: 10,834 řádků × 22 sloupců
Odstraněno sloupců: 57
  Z toho 100% NaN: license, neighbourhood_group_cleansed, calendar_updated
  Z toho ~59% NaN: neighborhood_overview, neighbourhood

99. percentil ceny: 21,823 CZK
Outlierů nad prahem: 95 (chybné záznamy nebo blokační ceny)
Celkem NaN v price_czk před imputací: 1431 (13.2 %)

Median před imputací: 2,136 CZK
Medián po imputaci:   2,086 CZK
✓ Imputace dokončena, NaN: 0

Zbývající NaN po čištění:
bathrooms_text               19
estimated_revenue_l365d    1336
host_since                   10
host_is_superhost           525

Průměrná vzdálenost od metra: 603 m
Medián vzdálenosti:           384 m
near_metro = True  (≤500 m):  7,144 (65.9 %)
near_metro = False (>500 m):  3,690 (34.1 %)

✓ listings_enriched.csv uložen: 10,834 řádků × 27 sloupců


---
## 2. Čištění calendar

In [27]:
if not os.path.exists(PROCESSED + 'calendar_clean.csv'):
    calendar = pd.read_csv(RAW + 'calendar.csv')
    print(f'Načteno: {calendar.shape[0]:,} řádků × {calendar.shape[1]} sloupců')

    # Odstranění sloupců s 100% NaN
    calendar.drop(columns=['price', 'adjusted_price'], inplace=True)
    print('Odstraněny sloupce: price, adjusted_price (100 % NaN)')

    # Konverze datumu
    calendar['date'] = pd.to_datetime(calendar['date'])

    # Odvození features
    calendar['occupied'] = (calendar['available'] == 'f').astype(int)
    calendar['month']    = calendar['date'].dt.month
    calendar['season']   = calendar['month'].map({
        12: 'zima',  1: 'zima',  2: 'zima',
         3: 'jaro',  4: 'jaro',  5: 'jaro',
         6: 'léto',  7: 'léto',  8: 'léto',
         9: 'podzim', 10: 'podzim', 11: 'podzim'
    })

    calendar.to_csv(PROCESSED + 'calendar_clean.csv', index=False)
    print(f'✓ calendar_clean.csv uložen: {calendar.shape[0]:,} řádků × {calendar.shape[1]} sloupců')
    print(f'Sloupce: {list(calendar.columns)}')
    print(f'Obsazenost celkem: {calendar["occupied"].mean()*100:.1f} %')
    print(f'Rozsah dat: {calendar["date"].min().date()} → {calendar["date"].max().date()}')

else:
    calendar = pd.read_csv(PROCESSED + 'calendar_clean.csv')
    print(f'calendar_clean.csv již existuje: {len(calendar):,} řádků — přeskakuji')

Načteno: 3,954,411 řádků × 7 sloupců
Odstraněny sloupce: price, adjusted_price (100 % NaN)
✓ calendar_clean.csv uložen: 3,954,411 řádků × 8 sloupců
Sloupce: ['listing_id', 'date', 'available', 'minimum_nights', 'maximum_nights', 'occupied', 'month', 'season']
Obsazenost celkem: 50.7 %
Rozsah dat: 2025-09-23 → 2026-09-23


---
## 3. Generování syntetických dat
### 3.1 bookings.csv

**Metodika:**
- Základ: consecutive dny `available = f` z calendar → jedna rezervace (run-length encoding)
- `guest_count`: useknuté normální rozdělení, moda=2, max=`accommodates` (reflektuje dominanci párů z EDA)
- `price_per_night`: ±15 % šum kolem reálné ceny z listings
- `status`: Cancelled/Confirmed — sezónně vážené dle EDA (zima + listopad vyšší cancel rate)

In [28]:
if not os.path.exists(SYNTHETIC + 'bookings.csv'):

    listings_enriched = pd.read_csv(PROCESSED + 'listings_enriched.csv')
    listing_info = listings_enriched.set_index('id')[['accommodates', 'price_czk']].to_dict('index')

    # Sezónní cancel rate — odvozeno z EDA obsazenosti
    CANCEL_RATE = {
        1: 0.20, 2: 0.20, 11: 0.20,   # zima/podzim — vysoké riziko
        3: 0.13, 4: 0.13, 5: 0.13,
        6: 0.10, 7: 0.10, 8: 0.10,    # léto — nejnižší riziko
        9: 0.10, 10: 0.13, 12: 0.18
    }
    
    MAX_STAY = 30  # max délka pobytu v dnech — krátkodobý pronájem

    def truncated_normal_guests(accommodates):
        """Useknuté normální rozdělení: moda=2, min=1, max=accommodates."""
        sample = int(np.round(np.random.normal(loc=2, scale=1.2)))
        return max(1, min(sample, accommodates))

    def make_booking(listing_id, group, start, end):
        check_in  = group.iloc[start]['date']
        check_out = (pd.to_datetime(group.iloc[end]['date']) + timedelta(days=1)).strftime('%Y-%m-%d')
        length    = (pd.to_datetime(group.iloc[end]['date']) - pd.to_datetime(check_in)).days + 1
        length = min(length, MAX_STAY)  # cap na krátkodobý pronájem
        info      = listing_info.get(listing_id, {'accommodates': 2, 'price_czk': 2152})
        ppn       = round(info['price_czk'] * random.uniform(0.85, 1.15), 2)
        month     = pd.to_datetime(check_in).month
        guests    = truncated_normal_guests(info['accommodates'])
        status    = 'Cancelled' if random.random() < CANCEL_RATE.get(month, 0.13) else 'Confirmed'
        return {
            'booking_id':        str(uuid.uuid4()),
            'listing_id':        listing_id,
            'check_in':          check_in,
            'check_out':         check_out,
            'length_of_stay':    length,
            'guest_count':       guests,
            'price_per_night':   ppn,
            'total_price':       round(ppn * length, 2),
            'status':            status,
            'booking_created_at': (pd.to_datetime(check_in) - timedelta(days=random.randint(1, 60))).strftime('%Y-%m-%d')
        }

    # Run-length encoding: consecutive occupied days → booking
    cal_raw  = pd.read_csv(RAW + 'calendar.csv')
    bookings = []
    for listing_id, group in cal_raw.groupby('listing_id'):
        group = group.sort_values('date').reset_index(drop=True)
        busy  = (group['available'] == 'f').tolist()
        start = None
        for i, is_busy in enumerate(busy):
            if is_busy and start is None:
                start = i
            elif not is_busy and start is not None:
                bookings.append(make_booking(listing_id, group, start, i - 1))
                start = None
        if start is not None:
            bookings.append(make_booking(listing_id, group, start, len(group) - 1))

    df_bookings = pd.DataFrame(bookings)

    # Validace
    over = sum(
        row['guest_count'] > listing_info.get(row['listing_id'], {'accommodates': 99})['accommodates']
        for _, row in df_bookings.iterrows()
    )
    assert over == 0, 'guest_count překračuje accommodates!'

    cancel_all    = (df_bookings['status'] == 'Cancelled').mean()
    winter        = df_bookings[pd.to_datetime(df_bookings['check_in']).dt.month.isin([1, 2, 11])]
    cancel_winter = (winter['status'] == 'Cancelled').mean()
    summer        = df_bookings[pd.to_datetime(df_bookings['check_in']).dt.month.isin([6, 7, 8])]
    cancel_summer = (summer['status'] == 'Cancelled').mean()

    df_bookings.to_csv(SYNTHETIC + 'bookings.csv', index=False)
    print(f'  bookings.csv vygenerován: {len(df_bookings):,} rezervací')
    print(f'  Cancel rate celkem:       {cancel_all:.1%}')
    print(f'  Cancel rate zima/nov:     {cancel_winter:.1%}')
    print(f'  Cancel rate léto:         {cancel_summer:.1%}')
    print(f'  Průměrná délka pobytu:    {df_bookings["length_of_stay"].mean():.1f} dní')
    print(f'  Průměrný počet hostů:     {df_bookings["guest_count"].mean():.1f}')
    print(f'  Průměrná cena/noc:        {df_bookings["price_per_night"].mean():.0f} CZK')

else:
    df_bookings = pd.read_csv(SYNTHETIC + 'bookings.csv')
    print(f'bookings.csv již existuje: {len(df_bookings):,} rezervací — přeskakuji generování')

  bookings.csv vygenerován: 74,671 rezervací
  Cancel rate celkem:       14.6%
  Cancel rate zima/nov:     19.9%
  Cancel rate léto:         9.6%
  Průměrná délka pobytu:    7.9 dní
  Průměrný počet hostů:     2.0
  Průměrná cena/noc:        2859 CZK


In [29]:
print(df_bookings['length_of_stay'].describe())
print(df_bookings['length_of_stay'].value_counts().sort_index().head(20))

count    74671.000000
mean         7.891243
std          8.930420
min          1.000000
25%          3.000000
50%          4.000000
75%          8.000000
max         30.000000
Name: length_of_stay, dtype: float64
length_of_stay
1      4499
2     11043
3     15857
4     10803
5      6288
6      3868
7      3190
8      1687
9      1397
10     1175
11     1129
12      877
13      733
14      636
15      384
16      309
17      286
18      295
19      246
20      217
Name: count, dtype: int64


### 3.2 expenses.csv

**Metodika:**
- Pouze pro `status = Confirmed`
- `cleaning_cost`: nelineární škála podle `bedrooms` + šum ±15 %
- `maintenance_cost`: náhodná událost s pravděpodobností 5 %

In [30]:
if not os.path.exists(SYNTHETIC + 'expenses.csv'):

    listings_enriched = pd.read_csv(PROCESSED + 'listings_enriched.csv')
    bedrooms_map = listings_enriched.set_index('id')['bedrooms'].fillna(1).astype(int).to_dict()

    def cleaning_cost(bedrooms):
        """Nelineární škála úklidu + šum ±15 %."""
        base = {0: 600, 1: 900, 2: 1400, 3: 2000, 4: 2600}.get(
            min(bedrooms, 4), 2600 + (bedrooms - 4) * 400
        )
        return round(base * random.uniform(0.85, 1.15))

    confirmed = df_bookings[df_bookings['status'] == 'Confirmed'].copy()
    expenses  = []
    for _, row in confirmed.iterrows():
        beds  = bedrooms_map.get(row['listing_id'], 1)
        clean = cleaning_cost(beds)
        maint = random.randint(500, 3000) if random.random() < 0.05 else 0
        expenses.append({
            'expense_id':       str(uuid.uuid4()),
            'booking_id':       row['booking_id'],
            'listing_id':       row['listing_id'],
            'cleaning_cost':    clean,
            'maintenance_cost': maint,
            'total_expense':    clean + maint,
            'expense_date':     row['check_out']
        })

    df_expenses = pd.DataFrame(expenses)
    df_expenses.to_csv(SYNTHETIC + 'expenses.csv', index=False)
    print(f'✓ expenses.csv vygenerován: {len(df_expenses):,} záznamů')
    print(f'  Průměrné náklady/rezervace: {df_expenses["total_expense"].mean():.0f} CZK')
    print(f'  Z toho s údržbou (5 %):     {(df_expenses["maintenance_cost"] > 0).mean():.1%}')
    print(f'  Min/Max celkových nákladů:  {df_expenses["total_expense"].min()} / {df_expenses["total_expense"].max()} CZK')

else:
    df_expenses = pd.read_csv(SYNTHETIC + 'expenses.csv')
    print(f'expenses.csv již existuje: {len(df_expenses):,} záznamů — přeskakuji generování')

✓ expenses.csv vygenerován: 63,765 záznamů
  Průměrné náklady/rezervace: 1248 CZK
  Z toho s údržbou (5 %):     4.9%
  Min/Max celkových nákladů:  510 / 5974 CZK


---
## 4. Načtení do DuckDB
### 4.1 Schéma databáze

```
metro_stations ──► listings_enriched ──► bookings ──► expenses
                        │
                        └──────────────► calendar
```

| Tabulka | Typ dat | PK | FK |
|---------|---------|----|----|   
| metro_stations | reálná | station_id | — |
| listings | reálná | id | nearest_metro_id → metro_stations.station_id |
| calendar | reálná | listing_id + date | listing_id → listings.id |
| bookings | syntetická | booking_id | listing_id → listings.id |
| expenses | syntetická | expense_id | booking_id → bookings.booking_id |

In [31]:
con = duckdb.connect(DB_PATH)
print(f'✓ DuckDB připojen: {DB_PATH}')

# Odstranění existujících tabulek (idempotentní běh)
for tbl in ['expenses', 'bookings', 'calendar', 'listings', 'metro_stations']:
    con.execute(f'DROP TABLE IF EXISTS {tbl}')

# metro_stations
con.execute("""
    CREATE TABLE metro_stations AS
    SELECT
        station_id::INTEGER  AS station_id,
        name::VARCHAR        AS name,
        line::VARCHAR        AS line,
        latitude::DOUBLE     AS latitude,
        longitude::DOUBLE    AS longitude
    FROM read_csv_auto(?)
""", [RAW + 'metro_stations.csv'])
print(f'metro_stations:  {con.execute("SELECT COUNT(*) FROM metro_stations").fetchone()[0]} řádků')

# listings
con.execute("""
    CREATE TABLE listings AS
    SELECT
        id::BIGINT                         AS id,
        name::VARCHAR                      AS name,
        latitude::DOUBLE                   AS latitude,
        longitude::DOUBLE                  AS longitude,
        neighbourhood_cleansed::VARCHAR    AS neighbourhood_cleansed,
        room_type::VARCHAR                 AS room_type,
        accommodates::INTEGER              AS accommodates,
        bedrooms::INTEGER                  AS bedrooms,
        beds::INTEGER                      AS beds,
        price_czk::DOUBLE                  AS price_czk,
        minimum_nights::INTEGER            AS minimum_nights,
        availability_365::INTEGER          AS availability_365,
        estimated_occupancy_l365d::INTEGER AS estimated_occupancy_l365d,
        estimated_revenue_l365d::DOUBLE    AS estimated_revenue_l365d,
        number_of_reviews::INTEGER         AS number_of_reviews,
        review_scores_rating::DOUBLE       AS review_scores_rating,
        host_id::BIGINT                    AS host_id,
        host_is_superhost::BOOLEAN         AS host_is_superhost,
        instant_bookable::BOOLEAN          AS instant_bookable,
        nearest_metro_id::INTEGER          AS nearest_metro_id,
        nearest_metro_name::VARCHAR        AS nearest_metro_name,
        nearest_metro_line::VARCHAR        AS nearest_metro_line,
        metro_distance_m::INTEGER          AS metro_distance_m,
        near_metro::BOOLEAN                AS near_metro
    FROM read_csv_auto(?)
""", [PROCESSED + 'listings_enriched.csv'])
print(f'listings:        {con.execute("SELECT COUNT(*) FROM listings").fetchone()[0]:,} řádků')

# calendar
con.execute("""
    CREATE TABLE calendar AS
    SELECT
        listing_id::BIGINT      AS listing_id,
        date::DATE              AS date,
        available::VARCHAR      AS available,
        occupied::INTEGER       AS occupied,
        minimum_nights::INTEGER AS minimum_nights,
        maximum_nights::INTEGER AS maximum_nights,
        month::INTEGER          AS month,
        season::VARCHAR         AS season
    FROM read_csv_auto(?)
""", [PROCESSED + 'calendar_clean.csv'])
print(f'calendar:        {con.execute("SELECT COUNT(*) FROM calendar").fetchone()[0]:,} řádků')

# bookings
con.execute("""
    CREATE TABLE bookings AS
    SELECT
        booking_id::VARCHAR      AS booking_id,
        listing_id::BIGINT       AS listing_id,
        check_in::DATE           AS check_in,
        check_out::DATE          AS check_out,
        length_of_stay::INTEGER  AS length_of_stay,
        guest_count::INTEGER     AS guest_count,
        price_per_night::DOUBLE  AS price_per_night,
        total_price::DOUBLE      AS total_price,
        status::VARCHAR          AS status,
        booking_created_at::DATE AS booking_created_at
    FROM read_csv_auto(?)
""", [SYNTHETIC + 'bookings.csv'])
print(f'bookings:        {con.execute("SELECT COUNT(*) FROM bookings").fetchone()[0]:,} řádků')

# expenses
con.execute("""
    CREATE TABLE expenses AS
    SELECT
        expense_id::VARCHAR       AS expense_id,
        booking_id::VARCHAR       AS booking_id,
        listing_id::BIGINT        AS listing_id,
        cleaning_cost::INTEGER    AS cleaning_cost,
        maintenance_cost::INTEGER AS maintenance_cost,
        total_expense::INTEGER    AS total_expense,
        expense_date::DATE        AS expense_date
    FROM read_csv_auto(?)
""", [SYNTHETIC + 'expenses.csv'])
print(f'expenses:        {con.execute("SELECT COUNT(*) FROM expenses").fetchone()[0]:,} řádků')

✓ DuckDB připojen: ../database/prague_stay.duckdb
metro_stations:  61 řádků
listings:        10,834 řádků
calendar:        3,954,411 řádků
bookings:        74,671 řádků
expenses:        63,765 řádků


### 4.2 Validace referenční integrity

In [32]:
checks = {
    'bookings.listing_id → listings.id': """
        SELECT COUNT(*) FROM bookings b
        LEFT JOIN listings l ON b.listing_id = l.id
        WHERE l.id IS NULL
    """,
    'expenses.booking_id → bookings.booking_id': """
        SELECT COUNT(*) FROM expenses e
        LEFT JOIN bookings b ON e.booking_id = b.booking_id
        WHERE b.booking_id IS NULL
    """,
    'expenses.listing_id → listings.id': """
        SELECT COUNT(*) FROM expenses e
        LEFT JOIN listings l ON e.listing_id = l.id
        WHERE l.id IS NULL
    """,
    'calendar.listing_id → listings.id': """
        SELECT COUNT(*) FROM calendar c
        LEFT JOIN listings l ON c.listing_id = l.id
        WHERE l.id IS NULL
    """,
    'listings.nearest_metro_id → metro_stations.station_id': """
        SELECT COUNT(*) FROM listings l
        LEFT JOIN metro_stations m ON l.nearest_metro_id = m.station_id
        WHERE m.station_id IS NULL
    """
}

print('Validace referenční integrity:')
print('-' * 55)
all_ok = True
for check_name, query in checks.items():
    result = con.execute(query).fetchone()[0]
    status = '✓' if result == 0 else '✗'
    if result > 0:
        all_ok = False
    print(f'{status} {check_name}: {result} porušení')

print()
print('✓ Všechny FK kontroly prošly!' if all_ok else '✗ Nalezena porušení integrity!')

Validace referenční integrity:
-------------------------------------------------------
✓ bookings.listing_id → listings.id: 0 porušení
✓ expenses.booking_id → bookings.booking_id: 0 porušení
✓ expenses.listing_id → listings.id: 0 porušení
✓ calendar.listing_id → listings.id: 0 porušení
✓ listings.nearest_metro_id → metro_stations.station_id: 0 porušení

✓ Všechny FK kontroly prošly!


### 4.3 Přehled databáze

In [33]:
print('=== PŘEHLED TABULEK V DuckDB ===')
print(con.execute('SHOW TABLES').df().to_string(index=False))

print('\n-- RevPAR podle near_metro (základ KPI #1) --')
print(con.execute("""
    SELECT
        near_metro,
        COUNT(*)                                   AS pocet_objektu,
        ROUND(AVG(estimated_revenue_l365d), 0)     AS avg_revenue_czk,
        ROUND(MEDIAN(estimated_revenue_l365d), 0)  AS median_revenue_czk
    FROM listings
    WHERE estimated_revenue_l365d > 0
    GROUP BY near_metro
    ORDER BY near_metro DESC
""").df().to_string(index=False))

print('\n-- Cancel rate podle měsíce (základ KPI #2) --')
print(con.execute("""
    SELECT
        MONTH(check_in) AS mesic,
        COUNT(*) AS celkem,
        SUM(CASE WHEN status = 'Cancelled' THEN 1 ELSE 0 END) AS cancelled,
        ROUND(AVG(CASE WHEN status = 'Cancelled' THEN 1.0 ELSE 0.0 END) * 100, 1) AS cancel_rate_pct
    FROM bookings
    GROUP BY MONTH(check_in)
    ORDER BY mesic
""").df().to_string(index=False))

con.close()
print('\n✓ DuckDB uzavřen')

=== PŘEHLED TABULEK V DuckDB ===
          name
      bookings
      calendar
      expenses
      listings
metro_stations

-- RevPAR podle near_metro (základ KPI #1) --
 near_metro  pocet_objektu  avg_revenue_czk  median_revenue_czk
       True           5338         376071.0            276462.0
      False           2660         258488.0            169575.0

-- Cancel rate podle měsíce (základ KPI #2) --
 mesic  celkem  cancelled  cancel_rate_pct
     1    2710      543.0             20.0
     2    1847      342.0             18.5
     3    3337      419.0             12.6
     4    2566      320.0             12.5
     5    1980      284.0             14.3
     6    2236      211.0              9.4
     7     875       89.0             10.2
     8     557       51.0              9.2
     9   12982     1335.0             10.3
    10   21963     2886.0             13.1
    11   10413     2095.0             20.1
    12   13205     2331.0             17.7

✓ DuckDB uzavřen


---
## 5. Shrnutí Data Preparation

| Krok | Akce | Výsledek |
|------|------|----------|
| Výběr sloupců | 79 → 22 sloupců | Odstraněny irelevantní a 100% NaN sloupce |
| Čištění ceny | String → float, outliers → NaN | Práh: 99. percentil (~21 823 CZK) |
| Imputace ceny | 3-stupňová podle skupin | 0 NaN po imputaci |
| Metro vzdálenost | Haversine vektorizovaný | `metro_distance_m`, `near_metro`, `nearest_metro_id` |
| Calendar | Odstraněny price/adj_price, přidány features | `occupied`, `month`, `season` |
| bookings.csv | Run-length encoding + sezónní cancel rate | Syntetická, korelovaná s reálnými daty |
| expenses.csv | Nelineární škála podle bedrooms + šum ±15 % | Syntetická, korelovaná s listings |
| DuckDB | 5 tabulek, FK validace | 0 porušení referenční integrity |

**Pokračujeme v 03_kpi_analysis.ipynb**